# 🚀 ST-TBA-GAT MAPPO Training on Kaggle GPU
Decentralized Multi-Agent Reinforcement Learning for Industrial Hazard Evacuation.

**Prerequisites on Kaggle:**
1. In the right panel, select **Accelerator: GPU T4 x2** or **GPU P100**.
2. Turn **Internet: ON** in the Settings panel.
3. Run all cells below.

In [ ]:
# Cell 1: Check GPU availability
!nvidia-smi
import torch
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"Device: {torch.cuda.get_device_name(0)}")

In [ ]:
# Cell 2: Step into /kaggle/working, clone clean repository branch, and install dependencies
%cd /kaggle/working
!rm -rf /kaggle/working/LBP
!git clone -b marl-v2-training https://github.com/kitretsu2809/decentralized_industrial_hazard_evacuation.git /kaggle/working/LBP
%cd /kaggle/working/LBP
!pip install --quiet gymnasium pettingzoo networkx onnx onnxscript


In [ ]:
# Cell 3: Verify all 82 unit tests pass on Kaggle
!python -m unittest discover -s tests -p "test_*.py"

In [ ]:
# Cell 4: Train ST-TBA-GAT with 4-Stage Curriculum on CUDA (100 Episodes)
!python -m simulator.training.train_mappo \
    --episodes 100 \
    --min-evacuees 20 \
    --max-evacuees 600 \
    --curriculum \
    --device cuda \
    --save-dir /kaggle/working/trained_checkpoints \
    --eval

In [ ]:
# Cell 5: Run Causal Parity Benchmark on 200 Evacuees
!python -c "from simulator.training.train_mappo import run_benchmark; run_benchmark(policy_path='/kaggle/working/trained_checkpoints/best_policy.pt', num_runs=5, num_evacuees=200, device='cuda')"

In [ ]:
# Cell 6: Package trained checkpoints for download
import os
!tar -czvf /kaggle/working/st_tba_gat_kaggle_checkpoints.tar.gz -C /kaggle/working/trained_checkpoints .
print("\n✅ Checkpoints packaged successfully at /kaggle/working/st_tba_gat_kaggle_checkpoints.tar.gz")
print("Download this file from the right sidebar under 'Output'.")